# NB1 — Data Preparation

Loads raw WADI A2 CSVs, cleans column names, parses timestamps, assigns labels, drops uninformative columns, performs stratified 80/20 train/test split on 30-second windows, and saves `data/wadi_prepared.parquet`.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path(".")
RAW_DIR     = PROJECT_DIR / "WaDi.A2_19 Nov 2019" / "WADI.A2_19 Nov 2019"
DATA_DIR    = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
WINDOW_SIZE = 30  # seconds per window for stratified split
TEST_RATIO  = 0.20

print(f"Raw data: {RAW_DIR}")
print(f"Output:   {DATA_DIR}")


Raw data: WaDi.A2_19 Nov 2019/WADI.A2_19 Nov 2019
Output:   data


## 1. Load & Clean Raw CSVs

In [2]:
def clean_colname(c: str) -> str:
    """Strip Windows path prefix from WADI sensor column names."""
    c = c.strip()
    if '\\' in c:
        return c.rsplit('\\', 1)[-1].strip()
    return c

def load_and_stage(path: Path, skiprows: int = 0, header: int = 0) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=skiprows, header=header, low_memory=False)
    df.columns = [clean_colname(c) for c in df.columns]
    # Drop unnamed / row-number columns (including "Row" — a CSV index counter, not a sensor)
    df = df.drop(columns=[c for c in df.columns
                           if not c or c.startswith("Unnamed") or c == "Row"],
                 errors="ignore")
    # Parse timestamp
    date_parsed = pd.to_datetime(df["Date"], format="mixed", dayfirst=False)
    time_parsed = pd.to_timedelta("00:" + df["Time"].fillna("00:00.0").astype(str))
    df["timestamp"] = (date_parsed + time_parsed).dt.tz_localize("UTC")
    df = df.drop(columns=["Date", "Time"], errors="ignore")
    return df

print("Loading normal operations data...")
df_normal = load_and_stage(RAW_DIR / "WADI_14days_new.csv")
print(f"  Shape: {df_normal.shape}")

print("\nLoading attack data...")
df_attack_raw = load_and_stage(RAW_DIR / "WADI_attackdataLABLE.csv", header=1)
print(f"  Shape: {df_attack_raw.shape}")
label_col = [c for c in df_attack_raw.columns if "LABLE" in c or "LABEL" in c][0]
print(f"  Label column: {label_col!r}")
print(df_attack_raw[label_col].value_counts().sort_index())


Loading normal operations data...
  Shape: (784571, 128)

Loading attack data...
  Shape: (172803, 129)
  Label column: 'Attack LABLE (1:No Attack, -1:Attack)'
Attack LABLE (1:No Attack, -1:Attack)
-1      9977
 1    162826
Name: count, dtype: int64


## 2. Assign Labels & Combine

In [3]:
df_normal = df_normal.copy()
df_normal["label"] = np.int8(0)

df_attack = df_attack_raw.copy()
label_col = [c for c in df_attack.columns if "LABLE" in c or "LABEL" in c][0]
df_attack["label"] = df_attack[label_col].apply(
    lambda x: np.int8(1) if x == -1.0 else np.int8(0)
)
df_attack = df_attack.drop(columns=[label_col])

df = pd.concat([df_normal, df_attack], ignore_index=True)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Combined shape: {df.shape}")
print(f"Time range:     {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"\nLabel counts:")
print(f"  Normal (0): {(df['label']==0).sum():>9,}")
print(f"  Attack (1): {(df['label']==1).sum():>9,}")


Combined shape: (957374, 129)
Time range:     2017-09-25 00:00:00+00:00 → 2017-10-11 00:59:59+00:00

Label counts:
  Normal (0):   947,397
  Attack (1):     9,977


## 3. Drop Uninformative Columns

In [4]:
meta_cols = {"timestamp", "label"}
candidates = [c for c in df.columns if c not in meta_cols]

null_counts = df[candidates].isnull().sum()
fully_null  = null_counts[null_counts == len(df)].index.tolist()
constant    = [c for c in candidates if c not in fully_null
               and df[c].nunique(dropna=True) <= 1]

drop_cols = fully_null + constant
print(f"Dropping {len(fully_null)} fully-null columns and {len(constant)} constant columns")
print(f"Total dropped: {len(drop_cols)}")

df = df.drop(columns=drop_cols)

SENSOR_COLS = [c for c in df.columns if c not in meta_cols]
print(f"\nSensor columns retained: {len(SENSOR_COLS)}")

# Save sensor column reference
(DATA_DIR / "sensor_cols.json").write_text(
    json.dumps({"sensor_cols": SENSOR_COLS, "n": len(SENSOR_COLS)}, indent=2)
)
print(f"Saved sensor_cols.json ({len(SENSOR_COLS)} sensors)")


Dropping 4 fully-null columns and 25 constant columns
Total dropped: 29

Sensor columns retained: 98
Saved sensor_cols.json (98 sensors)


## 4. Stratified Train/Test Split (30s Windows)

In [5]:
# Assign window IDs based on 30-second blocks
df["window_id"] = (df.index // WINDOW_SIZE).astype(int)
df["window_label"] = df.groupby("window_id")["label"].transform("max")

# Unique windows
windows = df[["window_id", "window_label"]].drop_duplicates().set_index("window_id")
n_windows = len(windows)
print(f"Total 30s windows: {n_windows:,}")
print(f"  Attack windows:  {(windows['window_label']==1).sum():,}")
print(f"  Normal windows:  {(windows['window_label']==0).sum():,}")

# Stratified random assignment
rng = np.random.default_rng(RANDOM_SEED)
split_map = {}
for label_val in [0, 1]:
    wids = windows.index[windows["window_label"] == label_val].tolist()
    rng.shuffle(wids)
    n_test = int(len(wids) * TEST_RATIO)
    for wid in wids[:n_test]:
        split_map[wid] = "test"
    for wid in wids[n_test:]:
        split_map[wid] = "train"

df["split"] = df["window_id"].map(split_map)
df = df.drop(columns=["window_id", "window_label"])

print(f"\nSplit assignment:")
for split in ["train", "test"]:
    mask = df["split"] == split
    print(f"  {split:<6}: {mask.sum():>9,} rows  "
          f"(normal={((df['split']==split)&(df['label']==0)).sum():,}  "
          f"attack={((df['split']==split)&(df['label']==1)).sum():,})")


Total 30s windows: 31,913
  Attack windows:  4,520
  Normal windows:  27,393

Split assignment:
  train :   765,914 rows  (normal=757,890  attack=8,024)
  test  :   191,460 rows  (normal=189,507  attack=1,953)


## 5. Save

In [6]:
out_path = DATA_DIR / "wadi_prepared.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:6]} ... ({len(df.columns)} total)")
print(f"Completed: {datetime.now()}")


Saved: data/wadi_prepared.parquet
Shape: (957374, 101)
Columns: ['1_AIT_001_PV', '1_AIT_002_PV', '1_AIT_003_PV', '1_AIT_004_PV', '1_AIT_005_PV', '1_FIT_001_PV'] ... (101 total)
Completed: 2026-04-19 10:50:48.265739
